In [ ]:
"""_summary_

✅ 실험 로그 (1회 실험당 1개 기록)
Experiment ID: exp_2026-07-05_ensemble_v3
날짜/시간: 2026-07-05
담당자: 김다혜
목표(가설): Tune 최적화된 여러 모델(v8n, v11n)을 WBF 앙상블로 합치면 단일 모델보다 mAP가 향상될 것
변경사항 요약(한 줄): YOLOv8n(Tune) + YOLOv11n(Tune) + YOLOv11n(Tune2+오버샘플링) WBF 앙상블
데이터
데이터 버전/커밋: 캐글 원본 데이터 (220장, 12건 제외)
Split: Train/Val/Test = 176장/44장 (8:2)
전처리: 976x1280 -> 패딩(1280x1280) -> 리사이즈(640x640)
증강(Aug):
1. v8n(Tune1) : HSV(h=0.0171, s=0.605, v=0.449) + Mosaic(0.852)
2. v11n(Tune1): 위 v8n과 동일한 Tune1 하이퍼파라미터 적용, 오버샘플링 미적용 (원본 176장)
3. v11n(Tune2): HSV + Mosaic + Flip(0.336) + translate(0.164) + scale(0.528), 오버샘플링(10개 기준) 데이터셋 사용

* Tune1 vs Tune2 차이:
  - Tune1: 오버샘플링 적용 전 원본 데이터(176장) 기준으로 model.tune() 1차 실행.
    HSV+Mosaic 증강 고정한 상태에서 lr, momentum, box/cls/dfl loss 가중치 등 
    30개 조합 자동 탐색 (iterations=30, epochs=30)
  - Tune2: 오버샘플링 적용 후 데이터셋(371장) 기준으로 model.tune() 2차 실행.
    증강 파라미터까지 전부 자동 탐색 범위에 포함시켜 재실행
    (translate, scale, flip 비율까지 자동으로 찾음)
  - 데이터셋과 탐색 범위가 다른 두 tune 결과를 각각 학습에 적용해 
    서로 다른 특성을 가진 모델을 확보, 앙상블 다양성 확보 목적

모델/학습 설정
모델: YOLOv8n + YOLOv11n × 2 (앙상블)
입력 크기 / Batch / Epoch: 640×640 / 16 / 100 (Patience 50)
Loss: YOLO 기본 손실함수 (CIoU + BCE + DFL)
Optimizer / lr / wd: AdamW / lr0=0.00176~0.00229 / wd=0.0
Scheduler: YOLO 기본 (cosine)
기타 핵심 하이퍼파라미터: model.tune()으로 탐색한 하이퍼파라미터 개별 적용
학습/검증 결과
mAP@0.5: 1.0000 (앙상블 val 기준)
mAP@0.5:0.95: 0.9883 (앙상블 val 기준)
Precision / Recall / F1: 0.9867 / 1.0000 / -
Conf/NMS threshold: conf=0.25, WBF iou_thr=0.5
학습 시간 / GPU: 각 모델 약 15~20분 (Colab T4)
에러 분석(필수)
실패 케이스 Top 3:
해당 없음 (val 기준 Recall 1.0000, FN 없음). 
다만 오검출(FP) 1건 존재 (conf=0.75 기준 정확 148개 중 1개), 
구체적 클래스 미확인으로 추가 분석 필요.
혼동 클래스:
- 클래스 간 혼동은 거의 없음
- 오검출 1건
대표 이미지/링크: (W&B run / 결과 이미지 경로)
결론
가설 채택/기각:
- 세 모델 앙상블이 단일 모델(mAP@50-95 최고 0.9840) 대비 0.9883으로 향상 확인, Kaggle 실제 제출 점수 0.95930
다음 액션: AI Hub 추가 데이터 병합 후 재학습
"""

In [ ]:
"""_summary_

✅ 실험 로그 (1회 실험당 1개 기록)
Experiment ID: exp_2026-07-06_aihub_combined
날짜/시간: 2026-07-06
담당자: 김다혜

목표(가설): 캐글 데이터(220장)만으로는 학습 데이터가 부족해 소수 클래스 성능이 낮았음. AI Hub 외부 데이터를 병합해 데이터 양을 늘리면 mAP 및 Kaggle 실제 점수가 
크게 향상될 것.

변경사항 요약(한 줄): 캐글(220장) + AI Hub 조합 데이터(10497장) 병합 → 118개 클래스로 확장, YOLOv8n 단일 모델 재학습

데이터
데이터 버전/커밋:
- 캐글 원본 데이터 (220장, 12건 제외 → train 176장)
- AI Hub 경구약제 조합 데이터 (TS_1,3,4,5,6,7,8, 10497장 성공/6장 실패)
- 총 train 10673장, val은 캐글 44장 그대로 유지 (AI Hub는 train에만 사용)

Split: Train/Val/Test = 10673장/44장
전처리: 976×1280 → 패딩(1280×1280) → 리사이즈(640×640), 캐글과 동일한 방식 적용
클래스 매핑: 팀원이 제공한 aihub_merged.json, class_map_118.json 사용.
캐글 56개 클래스(idx 0~55)와 AI Hub class_map의 0~55가 정확히 일치함을 확인 후 병합. AI Hub 전용 클래스는 idx 56~117(62개)로 확장 배정.

증강(Aug):
기존 model.tune() 1차 결과(Tune1) 하이퍼파라미터 재사용
- HSV(h=0.0171, s=0.605, v=0.449) + Mosaic(0.852)

모델: YOLOv8n (단일 모델, nc=118)
입력 크기 / Batch / Epoch: 640×640 / 16 / 60 (Patience 20)
Loss: YOLO 기본 손실함수 (CIoU + BCE + DFL)
Optimizer / lr / wd: AdamW / lr0=0.00176 / wd=0.0
Scheduler: YOLO 기본 (cosine)
기타 핵심 하이퍼파라미터: 기존 Tune 하이퍼파라미터(box=7.70337, cls=0.57286, dfl=2.02811) 그대로 적용

학습/검증 결과
mAP@0.5: 0.9950 (val 44장 기준)
mAP@0.5:0.95: 0.9923 (val 44장 기준, 기존 최고 단일모델 0.9840 대비 +0.0083)
Precision / Recall / F1: 0.9847 / 1.0000 / -
Conf/NMS threshold: conf=0.1~0.75 전 구간 동일 성능
학습 시간 / GPU: 약 4시간 (Colab T4, 로컬 복사 후 학습)

에러 분석(필수)
실패 케이스 Top 3:
해당 없음. val에 등장한 45개 클래스(캐글 56개 중) 전원 정확도 1.000 달성.
기존 베이스라인에서 FN이 다발했던 소수 클래스(알드린정, 써스펜8시간이알서방정, 엑스포지정, 트라젠타정 등)도 전부 100% 정확도로 개선됨.
오검출(FP) 0건.

혼동 클래스:
없음 (confusion matrix 대각선 외 영역 전부 0, 클래스 간 혼동 없음)

대표 이미지/링크:
- confusion_matrix_normalized.png (118개 클래스 기준)
- confidence_dist_aihub.png

결론
가설 채택/기각: 채택
- AI Hub 데이터 병합으로 학습 데이터가 48배(220장→10673장) 증가한 결과, val 기준 FN/FP가 완전히 사라지고 mAP@50-95가 0.9840 → 0.9923으로 향상
- Kaggle 실제 제출 점수: 0.95930(기존 앙상블 최고) → 0.99382로 대폭 상승
- 데이터 양 부족이 그동안 성능의 가장 큰 병목이었음을 확인

다음 액션:
- 병합 데이터(10673장) 기준으로 model.tune() 재실행하여 하이퍼파라미터 재탐색
- AI Hub 이미지 추가 확보 후 데이터 비교/보강
- 118개 클래스 중 AI Hub 전용 62개 클래스 성능 별도 검증
"""